In [ ]:
import os

from tqdm import tqdm

import jax
import jax.numpy as jnp
from jax import random, vmap

from jax.experimental import mesh_utils, multihost_utils
from jax.sharding import Mesh, PartitionSpec as P

from function_diffusion.models import Encoder, Decoder, DiT

from function_diffusion.utils.model_utils import (
    create_autoencoder_state,
    create_diffusion_state,
    create_optimizer,
    compute_total_params,
)
from function_diffusion.utils.train_utils import  sample_ode
from function_diffusion.utils.data_utils import create_dataloader
from function_diffusion.utils.checkpoint_utils import (
    create_checkpoint_manager,
    restore_checkpoint,
)

from model_utils import create_encoder_step, create_decoder_step
from burgers.data_utils import create_dataset

In [ ]:
from configs import diffusion

config = diffusion.get_config('fae,dit')

In [ ]:
def restore_fae_state(config, encoder, decoder):
    # Create learning rate schedule and optimizer
    lr, tx = create_optimizer(config)

    # Create train state
    state = create_autoencoder_state(config, encoder, decoder, tx)

    # Create checkpoint manager
    fae_job_name = f"{config.autoencoder.model_name}_use_pde_{config.training.use_pde}"

    ckpt_path = os.path.join(os.getcwd(), fae_job_name, "ckpt")
    ckpt_mngr = create_checkpoint_manager(config.saving, ckpt_path)

    # Restore the model from the checkpoint
    fae_state = restore_checkpoint(ckpt_mngr, state)
    print(f"Restored model {fae_job_name} from step", fae_state.step)

    return fae_state


In [ ]:
# Initialize function autoencoder
encoder = Encoder(**config.autoencoder.encoder)
decoder = Decoder(**config.autoencoder.decoder)

fae_state = restore_fae_state(config, encoder, decoder)

In [ ]:
# Initialize diffusion model
dit = DiT(**config.diffusion)
# Create learning rate schedule and optimizer
lr, tx = create_optimizer(config)

# Create diffusion train state
state = create_diffusion_state(config, dit, tx, use_conditioning=True)
num_params = compute_total_params(state)
print(f"Model storage cost: {num_params * 4 / 1024 / 1024:.2f} MB of parameters")

In [ ]:
# Create checkpoint manager
job_name = f"{config.diffusion.model_name}_use_pde_{config.training.use_pde}"
ckpt_path = os.path.join(os.getcwd(), job_name, "ckpt")
# Create checkpoint manager
ckpt_mngr = create_checkpoint_manager(config.saving, ckpt_path)

# Restore the model from the checkpoint
state = restore_checkpoint(ckpt_mngr, state)
print(f"Restored model {job_name} from step", state.step)

In [ ]:
# Device count
num_local_devices = jax.local_device_count()
num_devices = jax.device_count()
print(f"Number of devices: {num_devices}")
print(f"Number of local devices: {num_local_devices}")

# Create sharding for data parallelism
mesh = Mesh(mesh_utils.create_device_mesh((jax.device_count(),)), "batch")
state = multihost_utils.host_local_array_to_global_array(state, mesh, P())
fae_state = multihost_utils.host_local_array_to_global_array(fae_state, mesh, P())

In [ ]:
# Create encoder and decoder steps
encoder_step = create_encoder_step(encoder, mesh)
decoder_step = create_decoder_step(decoder, mesh)

In [ ]:
# Get test dataset
_, test_dataset = create_dataset(config)
test_loader = create_dataloader(test_dataset,
                                batch_size=4,
                                num_workers=config.dataset.num_workers,
                                shuffle=False)

In [ ]:
# Create uniform grid for evaluation
h, w = 200, 200

x_coords = jnp.linspace(0, 1, h)
y_coords = jnp.linspace(0, 1, w)
x_coords, y_coords = jnp.meshgrid(x_coords, y_coords, indexing='ij')
coords = jnp.hstack([x_coords.reshape(-1, 1), y_coords.reshape(-1, 1)])
coords = multihost_utils.host_local_array_to_global_array(coords, mesh, P())

In [ ]:
rng_key = jax.random.PRNGKey(888)

d = 1   # downsampling factor  # [1, 2, 5]
noise_level = 0.2

u_pred_list = []
u_true_list = []
r_pred_list = []
u_downsampled_list = []

iters = 0 
for batch in tqdm(test_loader):
    iters += 1
    rng_key, *keys = random.split(rng_key, 3)
    
    batch = jax.tree.map(jnp.array, batch)
    #u = batch
    u_downsammpled = batch[:, ::d, ::d]

    noise = random.normal(keys[0], u_downsammpled.shape) * 0.2 * noise_level
    u_downsammpled = u_downsammpled + noise

    #u_batch = (jnp.ones_like(u), u, jnp.ones_like(u))
    c_batch = (jnp.ones_like(u_downsammpled), u_downsammpled, jnp.ones_like(u_downsammpled))

    # # Shard the batch across devices
    """u_batch = multihost_utils.host_local_array_to_global_array(
        u_batch, mesh, P("batch")
        )"""
    c_batch = multihost_utils.host_local_array_to_global_array(
        c_batch, mesh, P("batch")
        )

    #z_u = encoder_step(fae_state.params[0], u_batch)
    z_c = encoder_step(fae_state.params[0], c_batch)

    z0 = random.normal(keys[1], shape=z_c.shape)

    z1_new, _ = sample_ode(state, z0=z0, c=z_c, num_steps=100, use_conditioning=True)  
    u_pred, r_pred = decoder_step(fae_state.params[1], z1_new, coords)

    u_pred = u_pred.reshape(-1, h, w)
    u_true = u.reshape(-1, h, w)
    r_pred = r_pred.reshape(-1, h, w)

    u = u.reshape(-1, h, w)
    u_downsammpled = u_downsammpled.reshape(-1, h//d, w//d)

    u_pred_list.append(u_pred)
    u_true_list.append(u)
    r_pred_list.append(r_pred)
    u_downsampled_list.append(u_downsammpled)

    if iters ==4:   # Comment out to run on full test set
        break

u_pred = jnp.vstack(u_pred_list)
u_true = jnp.vstack(u_true_list)
r_pred = jnp.vstack(r_pred_list)
u_downsammpled = jnp.vstack(u_downsampled_list)

In [ ]:
def compute_error(pred, y):
    return jnp.linalg.norm(pred.flatten() - y.flatten()) / jnp.linalg.norm(y.flatten())

error = vmap(compute_error)(u_pred, u_true)

print("Mean relative error:", jnp.mean(error))
print("Max relative error:", jnp.max(error))
print("Min relative error:", jnp.min(error))
print("Std relative error:", jnp.std(error))

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# ==================================================
# 输出目录
# ==================================================
vis_dir = os.path.join("./eval_results", "plots")
os.makedirs(vis_dir, exist_ok=True)

# ==================================================
# 辅助：将 JAX 数组安全转为 NumPy
# ==================================================
u_true_np = np.asarray(u_true)
u_pred_np = np.asarray(u_pred)
u_down_np  = np.asarray(u_downsammpled)
r_pred_np  = np.asarray(r_pred)

# 全局值范围（统一colorbar）
vmin_true = u_true_np.min()
vmax_true = u_true_np.max()
vmin_down = u_down_np.min()
vmax_down = u_down_np.max()
vmin_err = 0.0
vmax_err = np.abs(u_pred_np - u_true_np).max()
vmin_r = r_pred_np.min()
vmax_r = r_pred_np.max()

# ==================================================
# 1. 批量样本对比图（前 samples_plot 个）
# ==================================================
samples_plot = min(20, len(u_true_np))  # 可根据需要修改

for k in tqdm(range(samples_plot), desc="Saving sample plots"):
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))

    # Input (downsampled)
    im0 = axes[0].pcolor(u_down_np[k], cmap='jet', vmin=vmin_down, vmax=vmax_down)
    axes[0].set_title('Input (downsampled)')
    axes[0].set_aspect('equal')
    plt.colorbar(im0, ax=axes[0])

    # Reference
    im1 = axes[1].pcolor(u_true_np[k], cmap='jet', vmin=vmin_true, vmax=vmax_true)
    axes[1].set_title('Reference')
    axes[1].set_aspect('equal')
    plt.colorbar(im1, ax=axes[1])

    # Prediction
    im2 = axes[2].pcolor(u_pred_np[k], cmap='jet', vmin=vmin_true, vmax=vmax_true)
    axes[2].set_title('Prediction')
    axes[2].set_aspect('equal')
    plt.colorbar(im2, ax=axes[2])

    # Absolute Error
    error = np.abs(u_pred_np[k] - u_true_np[k])
    im3 = axes[3].pcolor(error, cmap='inferno', vmin=0, vmax=vmax_err)
    axes[3].set_title('Absolute Error')
    axes[3].set_aspect('equal')
    plt.colorbar(im3, ax=axes[3])

    # PDE residual
    im4 = axes[4].pcolor(r_pred_np[k], cmap='jet', vmin=vmin_r, vmax=vmax_r)
    axes[4].set_title('Predicted PDE Residual')
    axes[4].set_aspect('equal')
    plt.colorbar(im4, ax=axes[4])

    plt.tight_layout()
    plt.savefig(os.path.join(vis_dir, f"sample_{k:04d}.png"), dpi=300)
    plt.close()

# ==================================================
# 2. 误差分布分析图
# ==================================================
def plot_error_analysis(u_true_arr, u_pred_arr, save_path):
    """绘制误差分布统计：直方图、箱线图、散点图、空间平均误差图"""
    errors = (u_pred_arr - u_true_arr).flatten()
    # 随机采样用于散点图（最多10000点）
    idx = np.random.choice(len(errors), size=min(10000, len(errors)), replace=False)
    flat_true = u_true_arr.flatten()[idx]
    flat_pred = u_pred_arr.flatten()[idx]

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    # A. 误差直方图
    axes[0, 0].hist(errors, bins=100, alpha=0.7, color='blue', edgecolor='black')
    axes[0, 0].set_xlabel('Error')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].grid(True, alpha=0.3)
    mean_err = np.mean(errors)
    std_err = np.std(errors)
    axes[0, 0].axvline(mean_err, color='red', linestyle='--', label=f'Mean: {mean_err:.4f}')
    axes[0, 0].legend()

    # B. 误差箱线图
    axes[0, 1].boxplot(errors[idx])
    axes[0, 1].set_ylabel('Error')
    axes[0, 1].grid(True, alpha=0.3)

    # C. 真实 vs 预测散点图
    axes[1, 0].scatter(flat_true, flat_pred, alpha=0.2, s=2)
    min_val = min(flat_true.min(), flat_pred.min())
    max_val = max(flat_true.max(), flat_pred.max())
    axes[1, 0].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8)
    axes[1, 0].set_xlim(min_val, max_val)
    axes[1, 0].set_ylim(min_val, max_val)
    axes[1, 0].set_xlabel('True Values')
    axes[1, 0].set_ylabel('Predicted Values')
    axes[1, 0].set_aspect('equal')
    axes[1, 0].grid(True, alpha=0.3)

    # D. 平均绝对误差空间分布
    mean_abs_error = np.mean(np.abs(u_pred_arr - u_true_arr), axis=0)
    im = axes[1, 1].imshow(mean_abs_error, cmap='hot', origin='lower')
    axes[1, 1].set_xlabel('X')
    axes[1, 1].set_ylabel('Y')
    axes[1, 1].set_title('Mean Absolute Error Map')
    plt.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

plot_error_analysis(u_true_np, u_pred_np, os.path.join(vis_dir, "error_analysis.png"))

# ==================================================
# 3. 文本报告
# ==================================================
# 复用之前计算的误差指标（若不存在可重新计算）
report_path = os.path.join(vis_dir, "report.txt")
with open(report_path, "w") as f:
    f.write("Evaluation Report\n")
    f.write("=" * 30 + "\n")
    f.write(f"Number of samples: {len(u_true_np)}\n")
    f.write(f"Image size: {u_true_np.shape[1]}x{u_true_np.shape[2]}\n")
    f.write(f"Downsampling factor: d={d}\n")
    f.write(f"Noise level: {noise_level}\n")
    if 'error' in dir():
        # eval.ipynb 前面的 cell 已经计算了 error (vmap 相对误差)
        error_np = np.asarray(error)
        f.write(f"Relative L2 Error (mean ± std): {error_np.mean():.6f} ± {error_np.std():.6f}\n")
        f.write(f"Relative L2 Error (min / max): {error_np.min():.6f} / {error_np.max():.6f}\n")
    # 额外计算MAE、RMSE等
    mae = np.mean(np.abs(u_pred_np - u_true_np))
    rmse = np.sqrt(np.mean((u_pred_np - u_true_np)**2))
    f.write(f"Overall MAE: {mae:.6f}\n")
    f.write(f"Overall RMSE: {rmse:.6f}\n")
    f.write(f"Samples plotted: {samples_plot}\n")
    f.write(f"Plots saved to: {vis_dir}\n")

print(f"All visualizations saved to {vis_dir}")
print(f"Report written to {report_path}")